# Joining tables
Four genuinely different join shapes, because "a join" is not one thing.

**J1** is the everyday star-schema join: one big table, two small lookups. Small tables get
copied to every worker rather than shuffled, which is cheap. We deliberately raised Spark's
broadcast threshold in the settings so it does this properly — DuckDB does it automatically,
so leaving it off would have handicapped Spark.

**J2** is the one to watch. Both sides are large, so data genuinely has to move. This is the
case distributed engines were built for, and where the gap should be at its narrowest.

**J3** finds rows pointing at customers who do not exist. We planted about 1% of those on
purpose.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "03_joins", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")

In [ ]:
SQL_J1 = """
SELECT p.category, c.segment, count(*) AS orders, round(sum(s.amount),2) AS revenue
FROM sales s
JOIN products  p ON p.product_id  = s.product_id
JOIN customers c ON c.customer_id = s.customer_id
GROUP BY 1,2 ORDER BY revenue DESC
"""

# WHY THIS ONE:
#   The everyday join. Small tables get copied to workers rather than shuffled.
_, out, _ = bench.run(Case("J1", "Large joined to small (star schema)", "Joins", sql=SQL_J1.strip(),
                          why="The everyday join. Small tables get copied to workers rather than shuffled."))
display(out.head())

In [ ]:
SQL_J2 = """
SELECT r.reason, count(*) AS n, round(sum(r.refund_amount),2) AS refunded,
       round(avg(s.amount),2) AS avg_original
FROM sales s JOIN returns r ON r.sale_id = s.sale_id
GROUP BY 1 ORDER BY refunded DESC
"""

# WHY THIS ONE:
#   Both sides big, so data genuinely has to move. Spark's strongest case.
_, out, _ = bench.run(Case("J2", "Large joined to large (real shuffle)", "Joins", sql=SQL_J2.strip(),
                          why="Both sides big, so data genuinely has to move. Spark's strongest case."))
display(out.head())

In [ ]:
SQL_J3 = """
SELECT count(*) AS orphan_sales
FROM sales s LEFT JOIN customers c ON c.customer_id = s.customer_id
WHERE s.customer_id IS NOT NULL AND c.customer_id IS NULL
"""

# WHY THIS ONE:
#   Find records pointing at customers that do not exist. We planted ~1%.
_, out, _ = bench.run(Case("J3", "Anti-join (orphan rows)", "Joins", sql=SQL_J3.strip(),
                          why="Find records pointing at customers that do not exist. We planted ~1%."))
display(out.head())

In [ ]:
SQL_J4 = """
SELECT count(*) AS pairs FROM (
  SELECT a.sale_id FROM sales a
  JOIN sales b ON a.store_id = b.store_id AND a.sale_date = b.sale_date
   AND a.sale_id < b.sale_id AND a.channel = 'app' AND b.channel = 'app'
   AND a.region = 'north' AND b.region = 'north'
) t
"""

# WHY THIS ONE:
#   A table joined to itself. Expensive in any engine.
_, out, _ = bench.run(Case("J4", "Self-join (same store, same day)", "Joins", sql=SQL_J4.strip(),
                          why="A table joined to itself. Expensive in any engine."))
display(out.head())

In [ ]:
SQL_J5 = """
SELECT c.segment, count(*) AS active_customers
FROM customers c
WHERE EXISTS (SELECT 1 FROM sales s WHERE s.customer_id = c.customer_id AND s.amount > 100)
GROUP BY 1 ORDER BY 1
"""

# WHY THIS ONE:
#   'Which customers did X' without duplicating rows.
_, out, _ = bench.run(Case("J5", "Semi-join (customers who bought)", "Joins", sql=SQL_J5.strip(),
                          why="'Which customers did X' without duplicating rows."))
display(out.head())

## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()